# LAr optical map with RESuM

The objective is to emulate the **population-level detection fraction as a
function of position in liquid argon (LAr)**, reducing the need for expensive
optical simulations in population studies. The optical map estimates the average
simulated detection response at each position; individual-event classification
is a secondary diagnostic.

RESuM combines a **conditional neural process (CNP)** as a denoising step with a
**multi-fidelity Gaussian process (MFGP)** for spatial regression. The CNP learns
from low-fidelity (LF) simulations. The MFGP combines CNP-averaged predictions
with high-fidelity (HF) simulated detection fractions to estimate the HF optical
response and its predictive uncertainty.

This workflow uses LF non-zero-hit voxels, a two-output CNP, and class-aware
mixup with mixed context. The default loss is `theory-truth` Bernoulli NLL;
`LOSS_OVERRIDE = "practice-truth"` selects Gaussian NLL.

**Run All trains a fresh CNP and MFGP** and saves the results in a new folder.
Set `RUN_TRAINING = False` to inspect a saved run. Use the repository `.venv`
kernel and open the notebook from the repository root or `notebooks/` folder.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timezone
import json

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "1"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config.optical.resum.yaml").is_file())
sys.path.insert(0, str(ROOT))

import torch
from IPython.display import Image, Markdown, display
from schemas.surrogates import SurrogateRunConfig, load_surrogate_config
from core.surrogates.experiment import run_experiment
from data.optical_pipeline import load_prepared_batch

torch.set_num_threads(1)

## Settings

Architecture, training, mixup and MFGP settings come directly from
`config.optical.resum.yaml`. Its defaults are the two-output CNP,
`theory-truth` loss, class-aware mixup with `mix_context: true`, and the single
loss objective. Edit that file to change the experiment; the notebook only
assigns a fresh output folder and optionally overrides the loss.

In [ ]:
RUN_TRAINING = True
EXISTING_RUN = None  # Optional saved-run path; otherwise use the latest notebook run
LOSS_OVERRIDE = None  # Optional: "practice-truth" for Gaussian NLL

base = load_surrogate_config(ROOT / "config.optical.resum.yaml")
settings = base.model_dump(mode="python")
if LOSS_OVERRIDE is not None:
    settings["training"]["loss"] = LOSS_OVERRIDE
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
settings["output_directory"] = ROOT / "outputs/optical_resum/notebook" / run_id
config = SurrogateRunConfig.model_validate(settings)
assert config.mfgp is not None

if not RUN_TRAINING:
    completed = sorted((ROOT / "outputs/optical_resum/notebook").glob("*/mfgp/model.pkl"))
    EXISTING_RUN = Path(EXISTING_RUN or (completed[-1].parent.parent if completed else base.output_directory)).expanduser()
    if not EXISTING_RUN.is_absolute():
        EXISTING_RUN = ROOT / EXISTING_RUN

## Check the prepared data

Preparation is separate: this notebook uses saved normalized batches and split
metadata. It does not filter individual negative events or rebuild train/validation
splits. LF voxels must have at least one observed hit; HF zero-hit voxels are kept.
The CNP trains on LF only. The GP uses LF and HF training voxels. Test files are
never loaded. Excluding zero-hit LF voxels limits what that validation set represents.

In [ ]:
def show_table(rows, columns):
    def cell(value):
        return f"{value:.6g}" if isinstance(value, float) else str(value)
    text = "| " + " | ".join(columns) + " |\n"
    text += "| " + " | ".join(["---"] * len(columns)) + " |\n"
    text += "\n".join("| " + " | ".join(cell(row.get(c, "—")) for c in columns) + " |"
                      for row in rows)
    display(Markdown(text))

if RUN_TRAINING:
    counts = []
    for split in ("train", "validation"):
        for fidelity in ("lf", "hf"):
            batch = load_prepared_batch(config.data_directory / "batches" / split / f"{fidelity}.npz")
            zero = int((batch.labels.sum(axis=1) == 0).sum())
            if fidelity == "lf" and zero:
                raise ValueError("Expected LF non-zero-hit voxels; check data_directory.")
            counts.append(dict(split=split, fidelity=fidelity, voxels=batch.batch_size,
                               events_per_voxel=batch.n_events, hits=int(batch.labels.sum()),
                               zero_hit_voxels=zero))
    show_table(counts, ["split", "fidelity", "voxels", "events_per_voxel", "hits", "zero_hit_voxels"])

## Train the CNP and fit the MFGP

`run_experiment` executes the full pipeline:

1. Train the CNP from scratch and select the checkpoint by LF validation MAE.
2. Evaluate real LF/HF events and save CNP diagnostics and checkpoints.
3. Construct GP training levels: LF CNP means, HF CNP means, HF raw fractions.
4. Fit the GP and save predictions, normalization, plots, and the development map.

This is a synchronous run in the notebook kernel, on the configured device.
It does not submit a Slurm job. The prior CPU run took about four minutes;
runtime depends on the machine. The cell may remain quiet during fitting.

In [ ]:
RUN_DIR = run_experiment(config) if RUN_TRAINING else EXISTING_RUN

def read_json(path):
    return json.loads(path.read_text())

print("Results:", RUN_DIR)
print("Selected CNP step:", read_json(RUN_DIR / "checkpoints/best/model.json")["metadata"]["step"])
display(read_json(RUN_DIR / "experiment.json")["resolved_config"])

## CNP results on real events

Selection uses LF validation only. HF metrics are held-out reporting; training PR
is in-sample. The mean/residual plots show the selected and final checkpoints.

In [ ]:
def show_plot(path):
    display(Image(filename=str(path)))

cnp_metrics = read_json(RUN_DIR / "metrics.json")
show_table([r for r in cnp_metrics if r["checkpoint"] == "best" and r["split"] == "validation"],
           ["fidelity", "mean_observed", "mean_predicted", "voxel_rate_mae", "pearson_r", "average_precision"])
display(read_json(RUN_DIR / "sampling_audit.json"))
for name in ("validation_history", "lf_means", "hf_means", "lf_precision_recall", "hf_precision_recall"):
    show_plot(RUN_DIR / f"{name}.png")

## MFGP results and ±1σ/±2σ/±3σ coverage

The **HF highest-level output** estimates the simulation detection fraction.
The LF level is fitted to CNP means; comparing it with raw LF fractions also
reveals residual CNP bias. These are distinct diagnostics.

Coverage plots use the existing `viz.dispatch.plot_coverage_test` through the
shared pipeline. Bands include the GP's fitted observation noise; they are not
CNP uncertainty or pure latent-function bands. The Gaussian GP is unconstrained,
so negative predictions/bands are displayed without clipping. Ten HF validation
voxels provide only a preliminary coverage assessment.

In [ ]:
GP_DIR = RUN_DIR / "mfgp"
gp_metrics = read_json(GP_DIR / "metrics.json")
show_table(gp_metrics, ["fidelity", "level", "voxels", "mean_observed", "mean_predicted", "mae", "pearson_r"])
show_table([dict(fidelity=r["fidelity"], **{f"{k}σ": r["coverage"][str(k)] for k in (1, 2, 3)})
            for r in gp_metrics], ["fidelity", "1σ", "2σ", "3σ"])
for fidelity in ("hf", "lf"):
    show_plot(GP_DIR / f"{fidelity}_gp_means.png")
    show_plot(GP_DIR / f"{fidelity}_coverage.png")

## LAr optical map

The saved highest-fidelity map uses available development coordinates, not a
dense grid or test coordinates. All predictions and checkpoints remain saved
under `RUN_DIR`; the GP files are in its `mfgp/` subdirectory.

In [ ]:
show_plot(GP_DIR / "development_map.png")